# Week 08 — Home exercise 4: Ten companies through 2020

**Solution proposal.**

A loop over a folder, `concat`, the same aggregation written three ways so that the cost of each is
visible, and then the questions a table of ten share prices is actually for: who did well, who fell
furthest in the crash, and which days were unusual.

In [1]:
import os

import pandas as pd

## Task 1: read ten files into one table

Three things happen inside the loop, and only one of them is reading the file. The `ticker` column has
to be added **here**, while we still know which file the rows came from — once everything is
concatenated, that information is gone forever.

In [2]:
folder = "../data/stocks"

files = sorted(name for name in os.listdir(folder) if name.endswith(".csv"))

print(files)

['AAPL.csv', 'AMZN.csv', 'BABA.csv', 'FB.csv', 'GOOG.csv', 'JNJ.csv', 'JPM.csv', 'MSFT.csv', 'TSLA.csv', 'WMT.csv']


In [3]:
frames = []

for name in files:
    prices = pd.read_csv(os.path.join(folder, name))

    prices["Date"] = pd.to_datetime(prices["Date"], format="%Y-%m-%d")
    prices["ticker"] = name.split(".")[0]

    frames.append(prices)

stocks = pd.concat(frames).reset_index(drop=True)

print("rows:   ", len(stocks))
print("tickers:", stocks["ticker"].nunique())
stocks.head(3)

rows:    2520
tickers: 10


,Date,Open,High,Low,Close,Adj Close,Volume,ticker
0,2020-01-02,74.059998,75.150002,73.797501,75.087502,74.333511,135480400,AAPL
1,2020-01-03,74.287498,75.144997,74.125000,74.357498,73.610840,146322800,AAPL
2,2020-01-06,73.447502,74.989998,73.187500,74.949997,74.197395,118387200,AAPL


2 520 rows: ten companies times 252 trading days. The `reset_index(drop=True)` matters — without it
the table has ten separate runs of 0 to 251, and every `.loc` on it returns ten rows.

## Task 2: check it before trusting it

In [4]:
print("exact duplicate rows:        ", stocks.duplicated().sum())
print("repeated ticker-date keys:   ", stocks.duplicated(subset=["ticker", "Date"]).sum())
print("rows per ticker:")
print(stocks["ticker"].value_counts().head(3))

exact duplicate rows:         0
repeated ticker-date keys:    0
rows per ticker:
ticker
AAPL    252
AMZN    252
BABA    252
Name: count, dtype: int64


Both zero, and every ticker has the same 252 rows. The table is what it claims to be.

**Why both checks, when they agreed here?** Because they fail on different things.

- `duplicated()` catches the same *file* being read twice — a name listed twice, a cell re-run, a
  colleague's copy of January dropped into the folder. Every column matches, so the rows are exact
  copies and dropping them loses nothing.
- `duplicated(subset=["ticker", "Date"])` catches the same *company on the same day* appearing twice
  **with different prices**. That is not a copy of anything, so the first check says nothing at all.

The second finding something the first did not would be the more serious result. It would mean two
sources disagree about what Apple closed at on some given day — and no amount of dropping rows
answers that. You would have to find out which file is right, and probably ask whoever sent them.

## Task 3, version 1: with loops

No pandas grouping anywhere. The outer loop picks a ticker, the inner loop picks a month, and a filter
inside both narrows the table to one cell of the answer.

In [5]:
tickers = []
months = []
volumes = []

for ticker in sorted(stocks["ticker"].unique()):
    one_company = stocks[stocks["ticker"] == ticker]

    for month in range(1, 13):
        one_month = one_company[one_company["Date"].dt.month == month]

        tickers.append(ticker)
        months.append(month)
        volumes.append(one_month["Volume"].sum())

by_loop = pd.DataFrame({"ticker": tickers, "month": months, "volume": volumes})

print(by_loop.shape)
by_loop.head(3)

(120, 3)


,ticker,month,volume
0,AAPL,1,2934370400
1,AAPL,2,3019851200
2,AAPL,3,6280072400


Fifteen lines, three parallel lists that have to stay in step, and a filter that runs 120 times over
the whole table.

## Version 2: with `groupby`

In [6]:
by_groupby = (
    stocks
    .groupby(["ticker", stocks["Date"].dt.month])["Volume"]
    .sum()
    .reset_index()
)

by_groupby.columns = ["ticker", "month", "volume"]

print(by_groupby.shape)
by_groupby.head(3)

(120, 3)


,ticker,month,volume
0,AAPL,1,2934370400
1,AAPL,2,3019851200
2,AAPL,3,6280072400


`stocks["Date"].dt.month` is passed straight to `groupby` as a grouping key without ever becoming a
column — grouping by something you computed on the spot is allowed, and often tidier than adding a
column you only need once.

Grouping on the month *number* is safe here only because the file covers a single year. On two years
of data, January 2020 and January 2021 would land in the same group — the trap from the lecture.

## Version 3: with `resample`

In [7]:
by_resample = (
    stocks
    .set_index("Date")
    .groupby("ticker")
    .resample("ME")["Volume"]
    .sum()
    .reset_index()
)

print(by_resample.shape)
by_resample.head(3)

(120, 3)


,ticker,Date,Volume
0,AAPL,2020-01-31,2934370400
1,AAPL,2020-02-29,3019851200
2,AAPL,2020-03-31,6280072400


`groupby("ticker").resample("ME")` groups first by company and then resamples each company's rows on
the calendar — the same "group first" rule as everywhere else this week. Without the `groupby`, the
ten companies would be summed together into one series.

Note the month column: `2020-01-31`, a real date, rather than the integer `1`. That makes this the
version that would also be right on several years of data.

### Do all three agree?

In [8]:
by_resample["month"] = by_resample["Date"].dt.month

check = (
    by_loop
    .merge(by_groupby, on=["ticker", "month"], suffixes=("_loop", "_groupby"), validate="one_to_one")
    .merge(by_resample[["ticker", "month", "Volume"]], on=["ticker", "month"], validate="one_to_one")
)

print("rows:", len(check))
print("loop == groupby: ", (check["volume_loop"] == check["volume_groupby"]).all())
print("loop == resample:", (check["volume_loop"] == check["Volume"]).all())

rows: 120
loop == groupby:  True
loop == resample: True


All three identical on all 120 rows.

### Which would you write?

`groupby` or `resample`, without hesitating — and `resample` if there is any chance the data spans
more than one year.

The loop is not wrong, and it is worth having written once to see that `groupby` is not magic: it is
the loop, done for you. But count what it costs.

| | Lines | Places a mistake can hide |
|---|---|---|
| Loop | ~15 | Three lists that must stay in step; `range(1, 13)` (off by one at either end); the ticker list needing `sorted` for reproducibility; a month with no rows silently reported as zero |
| `groupby` | 6 | The grouping key — the month number only works within one year |
| `resample` | 7 | Forgetting the `groupby`, which would sum all ten companies together |

The three also behave differently when a month has **no rows** at all, which is worth knowing because
none of them warns you:

- The **loop** reports every month in `range(1, 13)` — a month with no data comes out as zero traded
  volume, which is a claim, not a gap.
- **`groupby`** only produces groups that have rows, so a missing month is simply absent.
- **`resample`** builds the months between each ticker's first and last row from the calendar. A gap
  in the middle appears as a row, and with `.sum()` its value is **zero**, not `NaN` — the sum of
  nothing. Months before a ticker's first row or after its last are not produced at all.

## Task 4: rebased to 100

`transform("first")` puts each ticker's first price beside every one of its rows — the baseline
pattern from the lecture, with the first value as the baseline. It only means "the first trading day"
because the table is sorted by date within ticker, so sort first.

In [9]:
stocks = stocks.sort_values(["ticker", "Date"])

first_price = stocks.groupby("ticker")["Adj Close"].transform("first")
stocks["rebased"] = 100 * stocks["Adj Close"] / first_price

stocks.groupby("ticker")["rebased"].agg(["min", "last"]).sort_values("last").round(1)

,min,last
ticker,,
JPM,56.4,92.1
BABA,80.2,108.5
JNJ,76.6,109.8
WMT,87.5,123.2
GOOG,77.3,127.2
FB,69.6,129.6
MSFT,84.5,139.5
AMZN,88.3,173.1
AAPL,74.9,179.6


**Tesla ended 2020 at 807 — up more than eightfold** — and nothing else comes close: Apple and Amazon
about three quarters up, Microsoft about 40%. At the other end, **JPMorgan is the only one of the ten
to finish below where it started**, and it had the deepest low of the year as well, at 56.

The `min` column says something the final column does not: every one of the ten lost more than a
tenth of its value at some point in the year, including the ones that finished far ahead.

Rebasing is what makes these comparable at all. Amazon's share price is in the thousands and Apple's
is under a hundred, so a chart of raw prices would show Amazon as a line across the top and the rest
as flat lines along the bottom.

## Task 5: the crash

Filtering on dates works exactly as filtering on anything else, because `Date` is a real date: the
strings in the mask are read as dates.

In [10]:
crash = stocks[(stocks["Date"] >= "2020-02-19") & (stocks["Date"] <= "2020-03-23")]

print("trading days in the window:", crash["Date"].nunique())

fall = crash.groupby("ticker")["Adj Close"].agg(["first", "last"])
fall["change_pct"] = 100 * (fall["last"] / fall["first"] - 1)

fall.sort_values("change_pct").round(1)

trading days in the window: 24


,first,last,change_pct
ticker,,,
TSLA,183.5,86.9,-52.7
JPM,131.8,75.8,-42.5
FB,217.5,148.1,-31.9
GOOG,1526.7,1056.6,-30.8
AAPL,80.3,55.7,-30.7
MSFT,185.4,134.6,-27.4
JNJ,144.1,108.2,-24.9
BABA,222.1,176.3,-20.6
AMZN,2170.2,1902.8,-12.3


**Tesla fell 53% and JPMorgan 43%; Amazon fell 12% and Walmart under 3%.**

Most of that is guessable once you think about what each company sells. A bank lends to businesses
that had just been told to close, so a recession hits it directly. Walmart sells groceries, which
people buy in a lockdown more rather than less, and Amazon delivers to people who cannot go to a shop.

Tesla is the one that does not fit a story about the business. It fell furthest in the crash and rose
furthest over the year, which is what a share does when its price rests on expectations about the
future rather than on this quarter's sales: it moves furthest in both directions.

## Task 6: unusual trading

Raw volume is a fact about company size. Dividing by each ticker's own median turns it into a fact
about the day: 2.0 means "twice a normal day's trading, for this company".

In [11]:
stocks["relative_volume"] = stocks["Volume"] / stocks.groupby("ticker")["Volume"].transform("median")
stocks["return_pct"] = stocks.groupby("ticker")["Adj Close"].pct_change() * 100

columns = ["ticker", "Date", "Volume", "relative_volume", "return_pct"]
stocks.sort_values("relative_volume", ascending=False)[columns].head(5).round(
    {"relative_volume": 1, "return_pct": 1}
)

,ticker,Date,Volume,relative_volume,return_pct
752,BABA,2020-12-24,141830000,8.9,-13.3
2433,WMT,2020-08-27,39672900,5.3,4.5
2436,WMT,2020-09-01,35708200,4.8,6.3
753,BABA,2020-12-28,73512100,4.6,0.2
754,BABA,2020-12-29,69715900,4.4,6.3


**Alibaba on 24 December 2020 traded almost nine times its normal volume, and fell 13%.** That was
the day Chinese regulators announced an antitrust investigation into the company — and Alibaba's two
other places in the list, on the following two trading days, are the same story continuing. The two Walmart
days at the end of August and start of September are its own news, at a time when it was bidding for
TikTok together with Microsoft and launching its Walmart+ membership.

Compare that with what raw volume would have said:

In [12]:
stocks.sort_values("Volume", ascending=False)[columns].head(5).round({"relative_volume": 1, "return_pct": 1})

,ticker,Date,Volume,relative_volume,return_pct
39,AAPL,2020-02-28,426884800,3.1,-0.1
48,AAPL,2020-03-12,418474000,3.0,-9.9
54,AAPL,2020-03-20,401693200,2.9,-6.3
146,AAPL,2020-07-31,374336800,2.7,10.5
49,AAPL,2020-03-13,370732000,2.7,12.0


**All five are Apple.** Apple is simply the most heavily traded of the ten, so a ranking of raw volume
is a ranking of company size and nothing else — the busiest day of the most unusual company never
appears. Measuring each row against its own group's typical level is the same move as the lecture's
baseline per country, and it is what turns "which numbers are biggest" into "which days were
different".

### Things worth noticing

- **`sorted()` on the file list is not decoration.** `os.listdir` returns names in whatever order the
  operating system feels like, which can differ between your machine and someone else's. An analysis
  whose output depends on that is not reproducible, and the fix costs six characters.
- **The filename was the only place the ticker existed.** Reading a folder of files is nearly always
  also a job of recording where each row came from, and the moment to do it is inside the loop.
- **Every per-company calculation needed the `groupby`.** `transform("first")`, `pct_change`,
  `resample` and the median would each have run straight across from one company into the next
  without it — ten series stacked in one table is exactly the lecture's panel, with tickers instead
  of countries.
- **The median, not the mean, as "a normal day".** The mean volume is pulled up by the very days we
  are looking for, so a day that is unusual would partly hide itself.

### What this notebook does NOT do

- It assumes all ten files have the same columns. `concat` would not complain if one of them did not —
  it would add the extra column and fill the other companies' rows with `NaN`. Checking
  `prices.columns` inside the loop would catch that.
- It never checks that each file covers the same dates. Ten files of 252 rows each is consistent with
  ten completely different years, and nothing here would notice.
- `Adj Close` is adjusted backwards from the date the data was downloaded, so its exact values depend
  on when that was. Rebased levels and returns are unaffected, which is why they are the numbers worth
  reporting.
- The `Volume` sums are compared with `==`, which is safe because volumes are whole numbers. On
  genuinely fractional results, two calculations done different ways can differ in the last decimal
  place, and `==` would report a disagreement that is not there.